In [11]:
import numpy as np

In [12]:
def line_search_backtracking(f_obj, grad, x, d, c1=1e-4, rho=0.5):
    """
    Búsqueda de línea por backtracking satisfaciendo la condición de Armijo.
    """
    alpha = 1.0
    g = grad(x)
    
    # Condición de Armijo: f(x + alpha*d) <= f(x) + c1 * alpha * (g^T d)
    while f_obj(x + alpha * d) > f_obj(x) + c1 * alpha * np.dot(g, d):
        alpha *= rho
        if alpha < 1e-12:  # Prevenir bucle infinito por alta rigidez
            break
            
    return alpha

def Gradiente_Conjugado_NL(f_obj, grad, x0, itermax=1000, tol=1e-6, method='PR'):
    """
    Algoritmo de Gradiente Conjugado No Lineal.
    
    Parámetros:
    -----------
    f_obj : función
        Función escalar objetivo f(x) a minimizar.
    grad : función
        Gradiente vectorial g(x) = grad(f(x)).
    x0 : array-like
        Vector o punto de estimación inicial.
    itermax : int
        Número máximo de iteraciones.
    tol : float
        Tolerancia basada en la norma del gradiente ||g(x)||.
    method : str
        Fórmula para beta: 'PR' (Polak-Ribière) o 'FR' (Fletcher-Reeves).
        
    Retorna:
    --------
    xk : ndarray
        Aproximación de la solución del problema.
    orbita : list
        Historial de los puntos xk visitados.
    error : float
        Norma del gradiente final ||g(xk)||.
    """
    x0 = np.array(x0, dtype=float).flatten()
    xk = x0.copy()
    orbita = [xk.copy()]
    
    gk = grad(xk)
    dk = -gk.copy()
    
    iter_count = 0
    
    while np.linalg.norm(gk) > tol and iter_count < itermax:
        # 1. Encontrar la longitud de paso alpha mediante búsqueda de línea
        alpha = line_search_backtracking(f_obj, grad, xk, dk)
        
        # 2. Actualizar la solución
        xk = xk + alpha * dk
        orbita.append(xk.copy())
        
        # 3. Evaluar el nuevo gradiente
        gk_next = grad(xk)
        
        # 4. Calcular el parámetro beta de conjugación
        if method == 'PR':
            # Polak-Ribière con reinicio automático (PR+)
            num = np.dot(gk_next, gk_next - gk)
            den = np.dot(gk, gk)
            beta = max(0.0, num / den)
        elif method == 'FR':
            # Fletcher-Reeves
            beta = np.dot(gk_next, gk_next) / np.dot(gk, gk)
        else:
            beta = 0.0
            
        # 5. Actualizar la dirección de búsqueda
        dk = -gk_next + beta * dk
        
        # 6. Reinicio a la dirección del gradiente si se pierde la propiedad de descenso
        if np.dot(gk_next, dk) >= 0:
            dk = -gk_next
            
        gk = gk_next
        iter_count += 1
        
    return xk, orbita, np.linalg.norm(gk)

In [14]:
# Función escalar objetivo de Rosenbrock: f(X) = 100*(X[1] - X[0]**2)**2 + (1 - X[0])**2
f = lambda X: 100.0 * (X[1] - X[0]**2)**2 + (1.0 - X[0])**2

# Gradiente analítico de la función de Rosenbrock
grad = lambda X: np.array([
    -400.0 * X[0] * (X[1] - X[0]**2) - 2.0 * (1.0 - X[0]),
    200.0 * (X[1] - X[0]**2)
])

x0 = np.array([0.2, 0.2])
xk, orbita, error = Gradiente_Conjugado_NL(f, grad, x0, itermax=1000, tol=1e-6, method='PR')

print('Solución:', xk)
print('Error:', error)
print('Iteraciones:', len(orbita) - 1)

Solución: [1.00000022 1.00000044]
Error: 9.704224151626894e-07
Iteraciones: 887
